# 07 — Closures and Decorators

Python functions are first-class objects. Closures remember variables from their defining scope, and decorators use that ability to add reusable behavior to functions.

*Intermediate · 75–120 min*

## You will learn

- Explain enclosing scope and nonlocal
- Write simple function factories
- Build a decorator using functools.wraps
- Use decorators without accidentally changing a function's contract

In [ ]:
# Setup: run this first. On Google Colab (or anywhere the course files are missing) it
# downloads the course and moves into this notebook's folder. Elsewhere it does nothing.
import os, sys
if not os.path.exists("../README.md"):
    course = os.path.join(os.path.expanduser("~"), "python-notebook")
    if not os.path.isdir(course):
        os.system(f"git clone -q --depth 1 https://github.com/SwarmLadder/python-notebook {course}")
    os.chdir(os.path.join(course, "intermediate", "notebooks"))
    sys.path.insert(0, os.getcwd())

## Examples

### 1. Functions returning functions

A closure captures variables from a surrounding function scope. A function factory can customize behavior without creating a whole class. Captured values remain accessible after the enclosing function has returned.

In [ ]:
def make_multiplier(factor):
    def multiply(value):
        return factor * value
    return multiply

double = make_multiplier(2)
triple = make_multiplier(3)
print(double(9), triple(9))

**Reasoning prompt:** Where does factor live after make_multiplier has finished?

### 2. nonlocal state

nonlocal allows rebinding a name from the nearest enclosing function scope. Use it for deliberately stateful closures. When state grows complicated, a class may be easier to understand.

In [ ]:
def make_counter():
    count = 0
    def next_count():
        nonlocal count
        count += 1
        return count
    return next_count

counter_a = make_counter()
counter_b = make_counter()
print(counter_a(), counter_a(), counter_b())

**Reasoning prompt:** Why do the two counters not share the same count?

### 3. Decorator mechanics and wraps

A decorator receives a function and returns a replacement callable. functools.wraps preserves the wrapped function's metadata, including its name and docstring. A decorator should propagate return values and errors unless designed otherwise.

In [ ]:
from functools import wraps

def trace_calls(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"calling {func.__name__}")
        return func(*args, **kwargs)
    return wrapper

@trace_calls
def add(a, b):
    """Return a + b."""
    return a + b

print(add(2, 5), add.__name__, add.__doc__)

**Reasoning prompt:** What goes wrong if wrapper forgets to return the function's result?

### 4. Parameterized decorators and validation

A decorator factory accepts configuration and returns the actual decorator. Nesting matters: factory → decorator → wrapper. Keep complicated validation close to domain logic if a decorator would obscure it.

In [ ]:
from functools import wraps

def require_minimum(minimum):
    def decorator(func):
        @wraps(func)
        def wrapper(value):
            if value < minimum:
                raise ValueError(f"must be at least {minimum}")
            return func(value)
        return wrapper
    return decorator

@require_minimum(0)
def square_root_approximation(value):
    return value ** 0.5

print(square_root_approximation(16))

**Reasoning prompt:** Can this particular wrapper accept arbitrary keyword arguments? How would you generalize it?

## Try it

Short fill-in exercises on this lesson: replace each `___`, run the cell, and `check` tells you whether you're right. The answer itself stays hidden. More practice for this topic is in the [workbook](../exercises/02_iteration_and_abstractions_exercises.ipynb).

In [ ]:
from lesson import Exercises, ___, run

check = Exercises('intermediate/07', {1: '475d5b7419a293d0', 2: '6c61da93644e702e', 3: 'd59a6caae02d2b52', 4: '6ef872ab7841712f'})

**Exercise 1.** A function factory: `make_multiplier(3)` returns a function that triples.

In [ ]:
def make_multiplier(factor):
    def multiply(x):
        return ___
    return ___

triple = make_multiplier(3)
check(1, [triple(2), make_multiplier(10)(5)])

**Exercise 2.** A closure that counts calls needs `nonlocal`.

In [ ]:
def make_counter():
    count = 0
    def counter():
        nonlocal count
        count += ___
        return ___
    return counter

c = make_counter()
check(2, [c(), c(), c()])

**Exercise 3.** A decorator that shouts the result, keeping the function's name with `functools.wraps`.

In [ ]:
import functools

def shout(func):
    @functools.___(func)
    def wrapper(*args, **kwargs):
        return ___.upper()
    return wrapper

@shout
def greet(name):
    return f"hello {name}"

check(3, (greet("ada"), greet.__name__))

**Exercise 4.** A decorator factory that repeats a function's result in a list.

In [ ]:
def repeat(times):
    def decorator(func):
        def wrapper(*args):
            return [func(*args) for _ in range(___)]
        return wrapper
    return decorator

@repeat(___)
def ping():
    return "ping"

check(4, ping())

## Common mistakes

- Use functools.wraps in decorators for metadata preservation.
- Avoid changing expected return values or swallowing errors accidentally.
- Use a class when stateful closure logic becomes hard to follow.

## Quick quiz

Replace each `'?'` in the cell below with your choice and run it. The answers are hidden, but each one is checked.

**1.** What does `nonlocal count` allow?

   - **a)** Assigning to a variable in the enclosing function
   - **b)** Reading a global
   - **c)** Creating a constant

**2.** Why use functools.wraps in a decorator?

   - **a)** Speed
   - **b)** It's required syntax
   - **c)** To keep the original __name__ and __doc__

**3.** @decorator above def f is the same as…

   - **a)** f = decorator(f)
   - **b)** decorator = f(decorator)
   - **c)** f(decorator)

In [ ]:
from quiz import Quiz

quiz = Quiz('intermediate/07', {1: '4892c802d94b', 2: '1b7a71ef164f', 3: 'b10d41e834f9'})
quiz.answer(1, '?')
quiz.answer(2, '?')
quiz.answer(3, '?')

**Next:** [08 — Context Managers and Resource Safety](08_context_managers_and_resources.ipynb)